# benchmark.ipynb — merenje performansi upita (v1 naivna vs v2 optimizovana)

"Pre/posle" poređenje istih upita nad dve verzije baze iz `data_load.ipynb`:

* **`{db}_v1`** — naivna: 4 odvojene kolekcije (`persons`, `products`, `orders`,
  `orderlines`), **sve vrednosti stringovi**, ravna polja, **bez indeksa**.
* **`{db}_v2`** — optimizovana: `orders` sa ugnježdenim `lines[]`, tipizirano
  (int/float/datetime), prirodni ključ kao `_id`, ciljani **indeksi**.

Za svaki upit i svaku verziju: (1) opciono očisti plan cache, (2) izmeri wall-clock
N pokretanja uz puno iscrpljivanje kursora (`list(cursor)`) i uzmi **medijanu** i **min**,
(3) jednom uzmi `explain` (`executionStats`) i rekurzivno izvuci `docsExamined`,
`keysExamined`, `nReturned`, `executionTimeMillis` i da li se IGDE javlja **COLLSCAN**.

Obe verzije svakog upita vraćaju **logički isti rezultat** — razlikuje se samo izvedba
(v1 kastuje stringove inline: `$toDouble`/`$toInt`/`substr` nad string datumom; v2 koristi
tipizirana polja, ugnježdeni `lines[]` i indekse).

> Upozorenje: naivni v1 upiti (npr. q8/q10 neindeksirani $lookup) mogu biti ekstremno
> spori. Pokreni sa `--timeout` (maxTimeMS) da bi se takvi slučajevi zabeležili kao
> `DNF/timeout` umesto da blokiraju — to je takođe validan nalaz za izveštaj.

**Notebook**: izmeni `NB_*` konstante u ćeliji *Konfiguracija* i pokreni sve ćelije.
**Skripta**: `jupyter nbconvert --to script benchmark.ipynb` pa
`python benchmark.py --timeout 60000 --runs 5 --warmup`.

In [83]:
"""Benchmark v1 (naivna) vs v2 (optimizovana) MongoDB šeme za projekat iz optimizacije upita.

Meri iste upite (q6–q10) nad bazama ``{db}_v1`` i ``{db}_v2``: medijana wall-clock vremena
kroz N pokretanja + sažetak ``explain`` (executionStats). Radi i kao Jupyter notebook
(NB_* konstante) i kao samostalna skripta (argparse CLI). Rezultati u ./bench_results:
results.json (sve), results.csv (sažetak po upitu), report.md (Markdown izveštaj).
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import sys
import time
from datetime import datetime
from statistics import median
from typing import Any, Callable, Optional

from pymongo import MongoClient
from pymongo.database import Database
from pymongo.errors import ExecutionTimeout, OperationFailure, PyMongoError

try:
    from tabulate import tabulate  # opciono, za lepšu poravnatu tabelu
except ImportError:  # fallback je ručno poravnanje kolona
    tabulate = None  # type: ignore[assignment]

## Konfiguracija (CLI + notebook fallback)

In [84]:
DEFAULT_URI = "mongodb://localhost:27017"
DEFAULT_DB = "ecommerce"

# Stvarne vrednosti iz dataseta (status je malim slovima).
ACTIVE_STATUS = "active"
DISCONTINUED_STATUS = "discontinued"
RATING_MIN = 4.5      # q6: "visok rating"
MIN_LINES = 6         # q9: porudžbine sa >= 6 stavki  ->  lines[MIN_LINES-1] postoji


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(
        description="Benchmark v1 (naivna) vs v2 (optimizovana) MongoDB šeme.")
    p.add_argument("--uri", default=DEFAULT_URI, help="MongoDB URI")
    p.add_argument("--db", default=DEFAULT_DB, help="meri nad {db}_v1 i {db}_v2")
    p.add_argument("--runs", type=int, default=5, help="broj pokretanja po upitu (medijana)")
    p.add_argument("--warmup", action="store_true",
                   help="odbaci prvo merenje (zagrevanje keša) pre medijane")
    p.add_argument("--clear-cache", action="store_true",
                   help="planCacheClear na ciljnoj kolekciji pre svakog upita")
    p.add_argument("--query", default=None, help="pokreni samo jedan upit, npr. q7")
    p.add_argument("--version", choices=["v1", "v2", "both"], default="both")
    p.add_argument("--out", default="./bench_results", help="izlazni direktorijum")
    p.add_argument("--timeout", type=int, default=0,
                   help="maxTimeMS po upitu (0 = bez limita)")
    return p


def _in_notebook() -> bool:
    return "ipykernel" in sys.modules or "ipykernel_launcher" in (sys.argv[0] if sys.argv else "")


# --- Notebook default-i (izmeni pri interaktivnom radu) ---
NB_URI = DEFAULT_URI
NB_DB = DEFAULT_DB
NB_RUNS = 1
NB_WARMUP = True
NB_CLEAR_CACHE = False
NB_QUERY: Optional[str] = None
NB_VERSION = "both"
NB_OUT = "./bench_results"
NB_TIMEOUT = 0  # preporuka: postavi npr. 60000 da naivni v1 upiti ne blokiraju


class Config:
    """Parametri pokretanja, jedinstveni za notebook i CLI."""

    def __init__(self, uri: str, db: str, runs: int, warmup: bool, clear_cache: bool,
                 query: Optional[str], version: str, out: str, timeout: int) -> None:
        self.uri = uri
        self.db = db
        self.runs = runs
        self.warmup = warmup
        self.clear_cache = clear_cache
        self.query = query
        self.version = version
        self.out = out
        self.timeout = timeout

    def __repr__(self) -> str:
        return (f"Config(uri={self.uri!r}, db={self.db!r}, runs={self.runs}, "
                f"warmup={self.warmup}, clear_cache={self.clear_cache}, query={self.query!r}, "
                f"version={self.version!r}, out={self.out!r}, timeout={self.timeout})")


def get_config(argv: Optional[list[str]] = None) -> Config:
    """Config iz NB_* (u notebook-u) ili iz CLI argumenata (kao skripta)."""
    if _in_notebook() and argv is None:
        return Config(NB_URI, NB_DB, NB_RUNS, NB_WARMUP, NB_CLEAR_CACHE,
                      NB_QUERY, NB_VERSION, NB_OUT, NB_TIMEOUT)
    a = build_parser().parse_args(argv)
    return Config(a.uri, a.db, a.runs, a.warmup, a.clear_cache,
                  a.query, a.version, a.out, a.timeout)

## Helperi za `explain`

`explain` se za `aggregate` i `find` razlikuje, a kod `$lookup` su metrike ugnježdene u
pod-pipeline. `walk_explain` rekurzivno prolazi ceo dict: sabira `docsExamined`/`keysExamined`
kroz **sve** pod-pipeline-ove, skuplja imena stage-ova i indeksa, beleži da li se IGDE
javlja `COLLSCAN`. Svi upiti ovde su agregacije, pa koristimo
`db.command("explain", {"aggregate": ...})`; helper `run_find_explain` pokriva i `find()`
slučaj (`cursor.explain()`) radi potpunosti.

In [85]:
def walk_explain(node: Any, acc: Optional[dict] = None) -> dict:
    """Rekurzivno prođi ceo explain dict i agregiraj metrike iz svih (pod)pipeline-ova."""
    if acc is None:
        acc = {"stages": [], "docs_examined": 0, "keys_examined": 0,
               "n_returned": [], "exec_ms": [], "indexes": [], "collscan": False}
    if isinstance(node, dict):
        for k, v in node.items():
            if k == "stage" and isinstance(v, str):
                acc["stages"].append(v)
                if v == "COLLSCAN":
                    acc["collscan"] = True
            elif k == "docsExamined" and isinstance(v, (int, float)):
                acc["docs_examined"] += v
            elif k == "keysExamined" and isinstance(v, (int, float)):
                acc["keys_examined"] += v
            elif k == "nReturned" and isinstance(v, (int, float)):
                acc["n_returned"].append(v)
            elif k in ("executionTimeMillis", "executionTimeMillisEstimate") and isinstance(v, (int, float)):
                acc["exec_ms"].append(v)
            elif k == "indexName" and isinstance(v, str):
                acc["indexes"].append(v)
            walk_explain(v, acc)
    elif isinstance(node, list):
        for item in node:
            walk_explain(item, acc)
    return acc


def _stage_label(acc: dict) -> str:
    """Kratak naziv winning plana: COLLSCAN ako se igde javlja, inače IXSCAN/FETCH/..."""
    if acc["collscan"]:
        return "COLLSCAN"
    scans = [s for s in acc["stages"] if s in ("IXSCAN", "FETCH", "COUNT_SCAN", "IDHACK", "DISTINCT_SCAN")]
    if scans:
        # zadrži redosled prvog pojavljivanja, bez duplikata
        seen: list[str] = []
        for s in scans:
            if s not in seen:
                seen.append(s)
        return "+".join(seen)
    return acc["stages"][0] if acc["stages"] else "?"


def summarize_explain(raw: dict) -> dict:
    """Sažmi sirovi explain u ravan rečnik metrika za izveštaj."""
    acc = walk_explain(raw)
    return {
        "status": "ok",
        "stage": _stage_label(acc),
        "collscan": acc["collscan"],
        "docs_examined": int(acc["docs_examined"]),
        "keys_examined": int(acc["keys_examined"]),
        "n_returned": max(acc["n_returned"]) if acc["n_returned"] else None,
        "execution_time_ms": max(acc["exec_ms"]) if acc["exec_ms"] else None,
        "indexes": sorted(set(acc["indexes"])),
    }


def run_explain(db: Database, coll_name: str, pipeline: list[dict], timeout_ms: int) -> dict:
    """explain(executionStats) za aggregate pipeline. Greške/timeout -> {status: DNF}."""
    cmd: dict[str, Any] = {
        "explain": {"aggregate": coll_name, "pipeline": pipeline,
                    "cursor": {}, "allowDiskUse": True},
        "verbosity": "executionStats",
    }
    if timeout_ms and timeout_ms > 0:
        cmd["maxTimeMS"] = timeout_ms
    try:
        raw = db.command(cmd)
    except (ExecutionTimeout, OperationFailure, PyMongoError) as exc:
        return {"status": "DNF", "error": f"{type(exc).__name__}: {exc}"}
    return summarize_explain(raw)


def run_find_explain(cursor, timeout_ms: int) -> dict:
    """explain za find() kursor (radi potpunosti — q6–q10 su agregacije)."""
    try:
        raw = cursor.explain()
    except (ExecutionTimeout, OperationFailure, PyMongoError) as exc:
        return {"status": "DNF", "error": f"{type(exc).__name__}: {exc}"}
    return summarize_explain(raw)

## Merenje (wall-clock) + plan cache

In [86]:
def clear_plan_cache(db: Database, coll_name: str) -> None:
    """planCacheClear na ciljnoj kolekciji (ignoriši ako nema keša)."""
    try:
        db.command("planCacheClear", coll_name)
    except PyMongoError:
        pass


def measure(db: Database, coll_name: str, pipeline: list[dict],
            runs: int, warmup: bool, timeout_ms: int) -> dict:
    """Pokreni aggregate N puta, meri wall-clock oko punog list(cursor). Vrati medijanu/min.

    Ako upit padne (timeout/veličina), zabeleži status 'DNF' sa porukom i prekini — to je
    validan nalaz za izveštaj.
    """
    kwargs: dict[str, Any] = {"allowDiskUse": True}
    if timeout_ms and timeout_ms > 0:
        kwargs["maxTimeMS"] = timeout_ms

    total_runs = runs + (1 if warmup else 0)
    times_ms: list[float] = []
    n_returned: Optional[int] = None
    for _ in range(total_runs):
        try:
            t0 = time.perf_counter()
            rows = list(db[coll_name].aggregate(pipeline, **kwargs))
            t1 = time.perf_counter()
        except (ExecutionTimeout, OperationFailure, PyMongoError) as exc:
            return {"status": "DNF", "error": f"{type(exc).__name__}: {exc}",
                    "times_ms": times_ms, "n_returned": n_returned}
        times_ms.append((t1 - t0) * 1000.0)
        n_returned = len(rows)

    measured = times_ms[1:] if (warmup and len(times_ms) > 1) else times_ms
    return {
        "status": "ok",
        "median_ms": median(measured),
        "min_ms": min(measured),
        "runs": len(measured),
        "warmup_dropped": bool(warmup and len(times_ms) > 1),
        "times_ms": times_ms,
        "n_returned": n_returned,
    }

## Registar upita (q6–q10)

Svaki upit ima `id`, `name`, `question` i dve implementacije: `build_v1(db)` i
`build_v2(db)`, koje vraćaju `(collection_name, pipeline)`. Obe daju **logički isti
rezultat** — razlikuje se samo izvedba. `build_*` prima `db` jer neki upiti (q6, q8)
prvo izvuku mali pomoćni skup ključeva pre glavnog pipeline-a.

Svaki upit je biran da pokaže konkretnu **tehniku optimizacije**: q6 — selektivni
indeks `lines.product_id`; q7 — opseg po indeksu `order_date` + embedding; q8 —
anti-join (indeksirani `distinct` + `$nin`); q9 — spoj preko `persons._id`; q10 —
embedding koji ukida spoj `orders⋈orderlines`.

In [ ]:
# =============================================================================
#  Registar upita q6–q10  —  primeri OPTIMIZACIJE UPITA (v1 naivna vs v2)
# -----------------------------------------------------------------------------
#  Isti ugovor: build_v1(db)/build_v2(db) -> (collection_name, pipeline).
#  Obe verzije vraćaju LOGIČKI ISTI rezultat; razlikuje se samo IZVEDBA.
#
#  ZAŠTO v1 PADA (DNF / više minuta), a v2 je brz:
#    v1 = 4 string kolekcije BEZ indeksa  ->  svaki "spoj" je NEINDEKSIRAN $lookup
#         nad 13M orderlines / 5M orders / 600k persons  =>  O(n*m)  =>  timeout.
#    v2 = ugnježdeni lines[] (nema orders⋈orderlines), tipovi i indeksi:
#         {order_date:1}, {lines.product_id:1}, persons._id, products._id.
# =============================================================================

# ---- Parametri (prilagodi stvarnim vrednostima u datasetu) ----
NB_LOW_STOCK = 5          # q6: prag "niske zalihe" (products.stock_quantity)
NB_REPORT_YEAR = 2024     # q7: godina za mesečni izveštaj prihoda
TOP_N = 20                # koliko redova vraćamo u top-listama


def _age_range_expr(age):
    """$switch: starost -> demografski opseg (age je IZRAZ, npr. "$age")."""
    return {"$switch": {"branches": [
        {"case": {"$lt":  [age, 18]}, "then": "<18"},
        {"case": {"$lte": [age, 25]}, "then": "18-25"},
        {"case": {"$lte": [age, 35]}, "then": "26-35"},
        {"case": {"$lte": [age, 45]}, "then": "36-45"},
        {"case": {"$lte": [age, 55]}, "then": "46-55"},
        {"case": {"$lte": [age, 65]}, "then": "56-65"},
    ], "default": "66+"}}


# =========================== q6 ==============================================
# (ZADRŽAN iz prethodne verzije, preimenovan q14 -> q6)
# Pitanje: "Koje proizvode HITNO dopuniti? (niska zaliha, a jaka realizovana tražnja)"
#   cover_ratio = stock / (units_sold + 1);  manji = hitnije.
# Sidro: products.stock_quantity (mali COLLSCAN 8k) -> lines.product_id (indeks v2).
def q6_build_v1(db):
    ids = [d["product_id"] for d in db.products.find(
        {"stock_quantity": {"$lte": str(NB_LOW_STOCK)}},   # v1: vrednosti su stringovi
        {"product_id": 1, "_id": 0})]
    return ("orderlines", [
        {"$match": {"product_id": {"$in": ids}}},
        {"$group": {"_id": "$product_id", "units_sold": {"$sum": {"$toInt": "$quantity"}}}},
        {"$lookup": {"from": "products", "localField": "_id",
                     "foreignField": "product_id", "as": "p"}},
        {"$unwind": "$p"},
        {"$project": {"_id": 0, "product_id": "$_id", "name": "$p.name",
                      "stock": {"$toInt": "$p.stock_quantity"}, "units_sold": 1,
                      "cover_ratio": {"$round": [{"$divide": [
                          {"$toInt": "$p.stock_quantity"},
                          {"$add": ["$units_sold", 1]}]}, 3]}}},
        {"$sort": {"cover_ratio": 1, "units_sold": -1}},
        {"$limit": TOP_N},
    ])


def q6_build_v2(db):
    ids = [d["_id"] for d in db.products.find(
        {"stock_quantity": {"$lte": NB_LOW_STOCK}}, {"_id": 1})]
    return ("orders", [
        {"$match": {"lines.product_id": {"$in": ids}}},
        {"$unwind": "$lines"},
        {"$match": {"lines.product_id": {"$in": ids}}},
        {"$group": {"_id": "$lines.product_id", "units_sold": {"$sum": "$lines.quantity"}}},
        {"$lookup": {"from": "products", "localField": "_id",
                     "foreignField": "_id", "as": "p"}},
        {"$unwind": "$p"},
        {"$project": {"_id": 0, "product_id": "$_id", "name": "$p.name",
                      "stock": "$p.stock_quantity", "units_sold": 1,
                      "cover_ratio": {"$round": [{"$divide": [
                          "$p.stock_quantity", {"$add": ["$units_sold", 1]}]}, 3]}}},
        {"$sort": {"cover_ratio": 1, "units_sold": -1}},
        {"$limit": TOP_N},
    ])


# =========================== q7 ==============================================
# Pitanje: "Mesečni prihod (sum subtotal) i broj porudžbina za datu godinu."
# Lekcija: opseg datuma preko indeksa {order_date:1} + EMBEDDING (subtotal u lines[])
#          vs. COLLSCAN nad STRING datumom + NEINDEKSIRAN spoj orders⋈orderlines.
def q7_build_v1(db):
    y = str(NB_REPORT_YEAR)
    return ("orders", [
        {"$addFields": {"y": {"$substr": ["$order_date", 0, 4]}}},
        {"$match": {"y": y}},                                    # COLLSCAN 5M (string datum)
        {"$lookup": {"from": "orderlines", "localField": "order_id",
                     "foreignField": "order_id", "as": "ol"}},    # NEINDEKSIRAN spoj nad 13M
        {"$addFields": {"month": {"$toInt": {"$substr": ["$order_date", 5, 2]}},
                        "total": {"$sum": {"$map": {"input": "$ol", "as": "l",
                                                    "in": {"$toDouble": "$$l.subtotal"}}}}}},
        {"$group": {"_id": "$month", "revenue": {"$sum": "$total"}, "n_orders": {"$sum": 1}}},
        {"$project": {"_id": 0, "month": "$_id",
                      "revenue": {"$round": ["$revenue", 2]}, "n_orders": 1}},
        {"$sort": {"month": 1}},
    ])


def q7_build_v2(db):
    start = datetime(NB_REPORT_YEAR, 1, 1)
    end = datetime(NB_REPORT_YEAR + 1, 1, 1)
    return ("orders", [
        {"$match": {"order_date": {"$gte": start, "$lt": end}}},    # IXSCAN {order_date:1}
        {"$project": {"month": {"$month": "$order_date"},
                      "total": {"$sum": "$lines.subtotal"}}},         # embedding: bez spoja
        {"$group": {"_id": "$month", "revenue": {"$sum": "$total"}, "n_orders": {"$sum": 1}}},
        {"$project": {"_id": 0, "month": "$_id",
                      "revenue": {"$round": ["$revenue", 2]}, "n_orders": 1}},
        {"$sort": {"month": 1}},
    ])


# =========================== q8 ==============================================
# Pitanje: "Demografski profil (opseg starosti, grad, pol) kupaca BEZ ijedne porudžbine."
# Lekcija: ANTI-JOIN. v1 = NEINDEKSIRAN $lookup persons⋈orders (600k × scan 5M) => DNF.
#          v2 = skup person_id koji IMAJU porudžbinu preko indeksa (DISTINCT_SCAN),
#               pa persons $match {_id: {$nin: ...}}  =>  milisekunde.
def q8_build_v1(db):
    return ("persons", [
        {"$lookup": {"from": "orders", "localField": "person_id",
                     "foreignField": "person_id", "as": "o"}},     # NEINDEKSIRAN spoj nad 5M
        {"$match": {"o": {"$eq": []}}},
        {"$addFields": {"age_range": _age_range_expr({"$toInt": "$age"})}},
        {"$group": {"_id": {"age_range": "$age_range", "city": "$city", "gender": "$gender"},
                    "count": {"$sum": 1}}},
        {"$project": {"_id": 0, "age_range": "$_id.age_range", "city": "$_id.city",
                      "gender": "$_id.gender", "count": 1}},
        {"$sort": {"count": -1}},
    ])


def q8_build_v2(db):
    # person_id koji IMAJU porudžbinu  ->  indeks {person_id, order_date} (DISTINCT_SCAN)
    with_orders = db.orders.distinct("person_id")
    return ("persons", [
        {"$match": {"_id": {"$nin": with_orders}}},
        {"$addFields": {"age_range": _age_range_expr("$age")}},
        {"$group": {"_id": {"age_range": "$age_range", "city": "$address.city",
                            "gender": "$gender"}, "count": {"$sum": 1}}},
        {"$project": {"_id": 0, "age_range": "$_id.age_range", "city": "$_id.city",
                      "gender": "$_id.gender", "count": 1}},
        {"$sort": {"count": -1}},
    ])


# =========================== q9 ==============================================
# Pitanje: "Top gradovi po ukupnoj potrošnji (sum total_amount) + prosek po kupcu."
# Lekcija: spoj orders→persons. v1 = NEINDEKSIRAN $lookup po person_id (scan 600k po
#          kupcu) => DNF; v2 = grupisanje pa $lookup po persons._id (indeks/IDHACK).
def q9_build_v1(db):
    return ("orders", [
        {"$group": {"_id": "$person_id", "spent": {"$sum": {"$toDouble": "$total_amount"}}}},
        {"$lookup": {"from": "persons", "localField": "_id",
                     "foreignField": "person_id", "as": "p"}},     # NEINDEKSIRAN spoj
        {"$unwind": "$p"},
        {"$group": {"_id": "$p.city", "revenue": {"$sum": "$spent"}, "customers": {"$sum": 1}}},
        {"$project": {"_id": 0, "city": "$_id", "revenue": {"$round": ["$revenue", 2]},
                      "customers": 1, "avg_per_customer": {"$round": [
                          {"$divide": ["$revenue", "$customers"]}, 2]}}},
        {"$sort": {"revenue": -1}},
        {"$limit": TOP_N},
    ])


def q9_build_v2(db):
    return ("orders", [
        {"$group": {"_id": "$person_id", "spent": {"$sum": "$total_amount"}}},
        {"$lookup": {"from": "persons", "localField": "_id",
                     "foreignField": "_id", "as": "p"}},            # persons._id indeks
        {"$unwind": "$p"},
        {"$group": {"_id": "$p.address.city", "revenue": {"$sum": "$spent"},
                    "customers": {"$sum": 1}}},
        {"$project": {"_id": 0, "city": "$_id", "revenue": {"$round": ["$revenue", 2]},
                      "customers": 1, "avg_per_customer": {"$round": [
                          {"$divide": ["$revenue", "$customers"]}, 2]}}},
        {"$sort": {"revenue": -1}},
        {"$limit": TOP_N},
    ])


# =========================== q10 =============================================
# Pitanje: "Po načinu plaćanja: ukupan prihod (sum subtotal), broj porudžbina i
#           prosečan broj stavki po porudžbini (prosečna korpa)."
# Lekcija: EMBEDDING ukida spoj. v1 = $group 13M po order_id pa NEINDEKSIRAN
#          $lookup orders (5M × scan 5M) => DNF; v2 = JEDAN prolaz nad orders.
def q10_build_v1(db):
    return ("orderlines", [
        {"$group": {"_id": "$order_id", "total": {"$sum": {"$toDouble": "$subtotal"}},
                    "n_lines": {"$sum": 1}}},
        {"$lookup": {"from": "orders", "localField": "_id",
                     "foreignField": "order_id", "as": "o"}},      # NEINDEKSIRAN spoj nad 5M
        {"$unwind": "$o"},
        {"$group": {"_id": "$o.payment_method", "revenue": {"$sum": "$total"},
                    "orders": {"$sum": 1}, "avg_basket": {"$avg": "$n_lines"}}},
        {"$project": {"_id": 0, "payment_method": "$_id",
                      "revenue": {"$round": ["$revenue", 2]}, "orders": 1,
                      "avg_basket": {"$round": ["$avg_basket", 2]}}},
        {"$sort": {"revenue": -1}},
    ])


def q10_build_v2(db):
    return ("orders", [
        {"$project": {"payment_method": 1, "n_lines": {"$size": "$lines"},
                      "total": {"$sum": "$lines.subtotal"}}},        # embedding: bez spoja
        {"$group": {"_id": "$payment_method", "revenue": {"$sum": "$total"},
                    "orders": {"$sum": 1}, "avg_basket": {"$avg": "$n_lines"}}},
        {"$project": {"_id": 0, "payment_method": "$_id",
                      "revenue": {"$round": ["$revenue", 2]}, "orders": 1,
                      "avg_basket": {"$round": ["$avg_basket", 2]}}},
        {"$sort": {"revenue": -1}},
    ])


# =============================================================================
#  Registar
# =============================================================================
QUERIES = [
    {"id": "q6", "name": "Hitna dopuna zaliha (niska zaliha vs tražnja)",
     "question": "Koje proizvode hitno dopuniti — niska zaliha, a jaka realizovana tražnja?",
     "build_v1": q6_build_v1, "build_v2": q6_build_v2},
    {"id": "q7", "name": "Mesečni prihod za godinu",
     "question": f"Ukupan prihod (sum subtotal) i broj porudžbina po mesecu za {NB_REPORT_YEAR}.",
     "build_v1": q7_build_v1, "build_v2": q7_build_v2},
    {"id": "q8", "name": "Profil kupaca bez narudžbina (anti-join)",
     "question": "Demografski profil (starost, grad, pol) kupaca bez ijedne porudžbine.",
     "build_v1": q8_build_v1, "build_v2": q8_build_v2},
    {"id": "q9", "name": "Top gradovi po potrošnji",
     "question": "Gradovi sa najvećom ukupnom potrošnjom i prosek po kupcu.",
     "build_v1": q9_build_v1, "build_v2": q9_build_v2},
    {"id": "q10", "name": "Prihod i prosečna korpa po načinu plaćanja",
     "question": "Po payment_method: prihod, broj porudžbina i prosečan broj stavki (korpa).",
     "build_v1": q10_build_v1, "build_v2": q10_build_v2},
]


## Pokretanje jednog upita (merenje + explain za obe verzije)

In [88]:
def run_query(db_v1: Database, db_v2: Database, q: dict, cfg: Config) -> dict:
    """Izmeri i objasni jedan upit za izabrane verzije."""
    res: dict[str, Any] = {"id": q["id"], "name": q["name"], "question": q["question"]}
    targets: list[tuple[str, Database, Callable]] = []
    if cfg.version in ("v1", "both"):
        targets.append(("v1", db_v1, q["build_v1"]))
    if cfg.version in ("v2", "both"):
        targets.append(("v2", db_v2, q["build_v2"]))

    for ver, db, build in targets:
        try:
            coll, pipeline = build(db)
        except PyMongoError as exc:  # npr. greška pri pripremi skupa product_id (q10)
            res[ver] = {"collection": None,
                        "measure": {"status": "DNF", "error": f"build: {exc}"},
                        "explain": {"status": "DNF", "error": f"build: {exc}"}}
            continue
        if cfg.clear_cache:
            clear_plan_cache(db, coll)
        m = measure(db, coll, pipeline, cfg.runs, cfg.warmup, cfg.timeout)
        ex = run_explain(db, coll, pipeline, cfg.timeout)
        res[ver] = {"collection": coll, "measure": m, "explain": ex}
        tag = (f"{m['median_ms']:.1f} ms (med), {m['n_returned']} red(ova)"
               if m["status"] == "ok" else f"DNF/timeout — {_short_err(m.get('error'))}")
        print(f"    {ver}: {tag}", flush=True)
    return res

## Izlaz: konzolna tabela, JSON, CSV, Markdown izveštaj

In [89]:
def _short_err(msg: Optional[str]) -> str:
    """Skrati dugačku pymongo poruku na suštinu (odbaci 'full error' dict)."""
    if not msg:
        return "DNF"
    return msg.split(", full error:")[0].strip()


def _fmt_ms(v: Optional[float]) -> str:
    return f"{v:,.1f}" if isinstance(v, (int, float)) else "DNF"


def _fmt_int(v: Optional[int]) -> str:
    return f"{v:,}" if isinstance(v, (int, float)) else "-"


def _median(res: dict, ver: str) -> Optional[float]:
    r = res.get(ver)
    if not r or r["measure"]["status"] != "ok":
        return None
    return r["measure"]["median_ms"]


def _explain_field(res: dict, ver: str, field: str) -> Any:
    r = res.get(ver)
    if not r or r["explain"].get("status") != "ok":
        return None
    return r["explain"].get(field)


def _speedup(res: dict) -> Optional[float]:
    v1, v2 = _median(res, "v1"), _median(res, "v2")
    if v1 is None or v2 is None or v2 == 0:
        return None
    return v1 / v2


def _cell_ms(res: dict, ver: str) -> str:
    d = res.get(ver)
    if not d:
        return "n/a"
    m = d["measure"]
    return _fmt_ms(m["median_ms"]) if m["status"] == "ok" else "DNF"


def _cell_explain(res: dict, ver: str, field: str, integer: bool = False) -> str:
    d = res.get(ver)
    if not d:
        return "n/a"
    ex = d["explain"]
    if ex.get("status") != "ok":
        return "DNF"
    return _fmt_int(ex.get(field)) if integer else (ex.get(field) or "-")


def _table_rows(results: list[dict]) -> list[list[str]]:
    rows = []
    for r in results:
        sp = _speedup(r)
        rows.append([
            r["id"],
            _cell_ms(r, "v1"),
            _cell_ms(r, "v2"),
            f"{sp:.1f}x" if sp is not None else "-",
            _cell_explain(r, "v1", "docs_examined", integer=True),
            _cell_explain(r, "v2", "docs_examined", integer=True),
            _cell_explain(r, "v1", "stage"),
            _cell_explain(r, "v2", "stage"),
        ])
    return rows


HEADERS = ["upit", "v1 med (ms)", "v2 med (ms)", "ubrzanje",
           "docs v1", "docs v2", "stage v1", "stage v2"]


def print_console_table(results: list[dict]) -> None:
    rows = _table_rows(results)
    print("\n===== REZULTATI =====")
    if tabulate is not None:
        print(tabulate(rows, headers=HEADERS, tablefmt="github",
                       colalign=("left", "right", "right", "right", "right", "right", "left", "left")))
        return
    # ručno poravnanje kolona ako tabulate nije instaliran
    cols = [HEADERS] + rows
    widths = [max(len(str(row[i])) for row in cols) for i in range(len(HEADERS))]
    for ridx, row in enumerate(cols):
        line = "  ".join(str(c).ljust(widths[i]) for i, c in enumerate(row))
        print(line)
        if ridx == 0:
            print("  ".join("-" * widths[i] for i in range(len(HEADERS))))


def _comment(res: dict) -> str:
    parts = []
    for ver in ("v1", "v2"):
        r = res.get(ver)
        if not r:
            continue
        m, ex = r["measure"], r["explain"]
        if m["status"] != "ok":
            parts.append(f"**{ver}**: DNF/timeout — {_short_err(m.get('error'))}")
            continue
        stage = ex.get("stage", "?") if ex.get("status") == "ok" else "?"
        docs = ex.get("docs_examined") if ex.get("status") == "ok" else None
        idx = ex.get("indexes") or []
        idx_txt = f" preko indeksa {', '.join(idx)}" if idx and not ex.get("collscan") else ""
        docs_txt = f"{docs:,} pregledanih dokumenata" if isinstance(docs, int) else "—"
        parts.append(f"**{ver}**: {stage}, {docs_txt}{idx_txt} "
                     f"({m['median_ms']:,.1f} ms med)")
    sp = _speedup(res)
    if sp is not None:
        parts.append(f"ubrzanje **{sp:.1f}x**")
    return "; ".join(parts)


def save_json(results: list[dict], out: str) -> None:
    payload = {"generated_at": datetime.now().isoformat(), "results": results}
    path = os.path.join(out, "results.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=str, ensure_ascii=False)
    print(f"  -> {path}")


def save_csv(results: list[dict], out: str) -> None:
    path = os.path.join(out, "results.csv")
    cols = ["query_id", "name",
            "v1_median_ms", "v1_min_ms", "v1_status", "v1_docs_examined",
            "v1_keys_examined", "v1_stage",
            "v2_median_ms", "v2_min_ms", "v2_status", "v2_docs_examined",
            "v2_keys_examined", "v2_stage", "speedup_v1_over_v2"]
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for r in results:
            row = [r["id"], r["name"]]
            for ver in ("v1", "v2"):
                d = r.get(ver)
                m = d["measure"] if d else {}
                ex = d["explain"] if d else {}
                ok = m.get("status") == "ok"
                exok = ex.get("status") == "ok"
                row += [
                    f"{m['median_ms']:.3f}" if ok else "",
                    f"{m['min_ms']:.3f}" if ok else "",
                    m.get("status", "missing"),
                    ex.get("docs_examined") if exok else "",
                    ex.get("keys_examined") if exok else "",
                    ex.get("stage") if exok else "",
                ]
            sp = _speedup(r)
            row.append(f"{sp:.3f}" if sp is not None else "")
            w.writerow(row)
    print(f"  -> {path}")


def save_report(results: list[dict], cfg: Config, total_s: float) -> None:
    path = os.path.join(cfg.out, "report.md")
    lines = [
        "# Benchmark izveštaj — v1 (naivna) vs v2 (optimizovana)",
        "",
        f"- Generisano: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"- Baze: `{cfg.db}_v1` vs `{cfg.db}_v2` @ `{cfg.uri}`",
        f"- Parametri: runs={cfg.runs}, warmup={cfg.warmup}, "
        f"clear_cache={cfg.clear_cache}, timeout={cfg.timeout} ms",
        f"- Ukupno trajanje benchmarka: {total_s:.1f} s",
        "",
        "## Rezime",
        "",
        "| " + " | ".join(HEADERS) + " |",
        "|" + "|".join(["---"] * len(HEADERS)) + "|",
    ]
    for row in _table_rows(results):
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    lines += ["", "## Komentar po upitu", ""]
    for r in results:
        lines.append(f"### {r['id']} — {r['name']}")
        lines.append(f"*{r['question']}*")
        lines.append("")
        lines.append(_comment(r))
        lines.append("")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"  -> {path}")

## Glavni runner

In [90]:
def main(argv: Optional[list[str]] = None) -> list[dict]:
    cfg = get_config(argv)
    print(cfg)
    if cfg.timeout == 0 and cfg.version != "v2":
        print("UPOZORENJE: --timeout 0 (bez limita). Naivni v1 upiti (npr. q7) mogu dugo "
              "da blokiraju. Razmisli o npr. --timeout 60000.", flush=True)

    client = MongoClient(cfg.uri)
    try:
        client.admin.command("ping")
    except Exception as exc:
        raise SystemExit(f"Ne mogu da se povežem na MongoDB ({cfg.uri}): {exc}")

    db_v1 = client[f"{cfg.db}_v1"]
    db_v2 = client[f"{cfg.db}_v2"]

    queries = QUERIES
    if cfg.query:
        queries = [q for q in QUERIES if q["id"] == cfg.query]
        if not queries:
            raise SystemExit(f"Nepoznat upit: {cfg.query!r}. Dostupni: "
                             f"{[q['id'] for q in QUERIES]}")

    t0 = time.perf_counter()
    results: list[dict] = []
    try:
        for q in queries:
            print(f"\n>>> {q['id']}: {q['name']}", flush=True)
            results.append(run_query(db_v1, db_v2, q, cfg))
    finally:
        client.close()
    total_s = time.perf_counter() - t0

    os.makedirs(cfg.out, exist_ok=True)
    print_console_table(results)
    print("\nSnimanje rezultata:")
    save_json(results, cfg.out)
    save_csv(results, cfg.out)
    save_report(results, cfg, total_s)
    print(f"\nUkupno trajanje benchmarka: {total_s:.1f} s")
    return results

## Pokretanje
U notebook-u koristi `NB_*` konstante; kao skripta parsira CLI argumente.

In [91]:
if __name__ == "__main__" or _in_notebook():
    _results = main()

Config(uri='mongodb://localhost:27017', db='ecommerce', runs=1, warmup=True, clear_cache=False, query=None, version='both', out='./bench_results', timeout=0)
UPOZORENJE: --timeout 0 (bez limita). Naivni v1 upiti (npr. q7) mogu dugo da blokiraju. Razmisli o npr. --timeout 60000.

>>> q6: Hitna dopuna zaliha (niska zaliha vs tražnja)
    v1: 55034.4 ms (med), 20 red(ova)
    v2: 2984.0 ms (med), 20 red(ova)

>>> q8: Profil kupaca bez narudžbina (anti-join)


KeyboardInterrupt: 